In [1]:
import os
import torchvision.models as models
from PIL import Image
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset
from torchvision.datasets.utils import download_and_extract_archive

from torch.utils.data import DataLoader
from torchvision import transforms

# Fine-tuning and bounding boxes
- místo trénování from scratch, použiju předtrénovaný model (pretrained) model a dotrénuji na několik málo epoch na svých datech
- freeze/unfreeze:
    už je natrénováno do K=1000 tříd, tak třeba přidáme něco že chceme jen jedno číslo (vyměnili jsme poslední layer a tak třeba freezneme a budeme trénovat jenom ten konec (třeba by se nám to ani nevešlo do paměti, kdybychom měli trénovat všechno) (dopisujeme jenom head, kterej je specifickej pro náš task)
- we take pre-trained model (e.g. ResNet) on image classification task (náš backbone)
- then we fine-tune the weights on bounding box task (každýho chodce dáme do obdelníku) (budeme chtít predikovat čtveřice čísel (hranice toho obdelníku))
- we use Penn-Fudan database (see https://www.cis.upenn.edu/~jshi/ped_html/)
- finally, we evaluate the model performance using standard metric
- TODOs: 1) Implement ResNetBoxes, 2) Choose loss function, 3) Implement IoU as a metric and evaluate performance, 4) Improve ResNetBoxes with argument that determines if the backbone network should be freezed for fine-tuning, 5) Observe performance when different ResNet variants are used

In [2]:
try: # disable certificate verification, needed on MacOS
    import ssl
    ssl._create_default_https_context = ssl._create_unverified_context
except ImportError:
    pass  # SSL module not available, skipping workaround

In [3]:
# Download and extract dataset
url = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"
download_and_extract_archive(url, download_root="data/", extract_root="data/", remove_finished=True)

100%|██████████| 53.7M/53.7M [00:09<00:00, 5.91MB/s]


Extracting data/PennFudanPed.zip to data/


In [21]:
class PennFudanDataset(Dataset):
    def __init__(self, root, transforms=None):
        self.root = root
        self.transforms = transforms
        self.img_dir = os.path.join(root, "PNGImages")  
        self.mask_dir = os.path.join(root, "PedMasks")
        self.imgs = sorted(os.listdir(self.img_dir))
        self.masks = sorted(os.listdir(self.mask_dir))

    def __getitem__(self, idx):
        # Construct full paths
        img_path = os.path.join(self.img_dir, self.imgs[idx])
        mask_path = os.path.join(self.mask_dir, self.masks[idx])

        # Load image and mask
        img = Image.open(img_path).convert("RGB")
        mask = np.array(Image.open(mask_path))

        # Remove background (assumed to be ID 0)
        obj_ids = np.unique(mask)
        obj_ids = obj_ids[obj_ids != 0]

        # Generate binary masks
        masks = mask == obj_ids[:, None, None]

        # Generate bounding boxes
        boxes = []
        for m in masks:
            pos = np.where(m)
            xmin, xmax = pos[1].min(), pos[1].max()
            ymin, ymax = pos[0].min(), pos[0].max()
            boxes.append([xmin, ymin, xmax, ymax])
        boxes = torch.tensor(boxes, dtype=torch.float32)
        orig_w, orig_h = img.size # škáluju, dopsala jsem já
        box = boxes[0] # škáluju, dopsala jsem já
        box[0] /= orig_w # škáluju, dopsala jsem já
        box[2] /= orig_w # škáluju, dopsala jsem já
        box[1] /= orig_h # škáluju, dopsala jsem já
        box[3] /= orig_h # škáluju, dopsala jsem já

        # All objects are labeled as class 1 (pedestrian)
        labels = torch.ones((len(obj_ids),), dtype=torch.int64)

        # Construct target dictionary
        target = {
            "box": boxes[0],  # First object's box only
            "label": labels[0],
        }

        if self.transforms:
            img = self.transforms(img)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [27]:
class ResNetBoxes(nn.Module):
    def __init__(self, resnet, freeze=False):
        super().__init__()
        # dopsano td 1: Use the backbone from `resnet` (all layers except the final FC).
        #         Add adaptive average pooling, a flatten layer, and a small head
        #         (e.g. Linear -> ReLU -> Linear) that outputs 4 bounding box coordinates.
        # Hint: resnet.fc.in_features gives the backbone's output channel count.
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) # resnet bez posledních částí avgpool + fc
        num_features = resnet.fc.in_features

        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        # regression head: výstup jsou 4 souřadnice boxu
        self.head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Linear(256, 4),
            nn.Sigmoid()   # prý důležité
        )
        ...
        # dopsano td 4: If `freeze` is True, set requires_grad = False for all backbone parameters.
        if freeze == True: # nebo jenom prej if freeze
            for param in self.backbone.parameters():
                param.requires_grad = False


    def forward(self, x):
        # dopsano td 1: Pass x through backbone -> pool -> head and return the 4-d output.
        x = self.backbone(x)
        x = self.pool(x)
        x = self.flatten(x)
        x = self.head(x)
        return x

In [28]:
# Dataset & loader
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])
dataset = PennFudanDataset(root='data/PennFudanPed', transforms=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(dataset, batch_size=32, shuffle=False) # toto dodělávám já, jestli je to dobře netuším

# Model & optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
backbone_model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
model = ResNetBoxes(backbone_model, freeze=True).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# dopsano td 2: Choose an appropriate loss function for bounding box regression.
criterion = nn.MSELoss()  # compare e.g. nn.SmoothL1Loss() and nn.MSELoss()

Intersection over Union (IoU) measures how well a predicted box overlaps with the ground truth:

$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

Each box is represented as `[xmin, ymin, xmax, ymax]`.

In [29]:
def compute_iou(box1, box2):
    """Compute IoU between two boxes [xmin, ymin, xmax, ymax]."""
    #dopsáno td 3: Implement this function.
    # Steps:
    #   1. Compute the coordinates of the intersection rectangle.
    x_leva = max(box1[0], box2[0])
    y_dolni = max(box1[1], box2[1])
    x_prava = min(box1[2], box2[2])
    y_horni = min(box1[3], box2[3])
    #   2. Compute its area (watch out for non-overlapping boxes).
    if x_prava <= x_leva or y_horni <= y_dolni:
        return 0
    intersection = (x_prava - x_leva)*(y_horni - y_dolni)

    #   3. Compute the area of each box.
    area1 = (box1[2] - box1[0])*(box1[3] - box1[1])
    area2 = (box2[2] - box2[0])*(box2[3] - box2[1])
    #   4. Return intersection_area / (area1 + area2 - intersection_area).
    union = area1 + area2 - intersection
    return intersection / union

In [ ]:
# Training loop
for epoch in range(5):
    model.train()
    running_loss = 0.0
    for imgs, targets in loader:
        imgs = imgs.to(device)
        gt_boxes = targets["box"].to(device)

        preds = model(imgs)
        loss = criterion(preds, gt_boxes)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    #dopsáno: add evaluation on test dataset using our 'compute_iou'
    # evaluation
    model.eval()
    running_iou = 0.0

    with torch.no_grad(): # vypne počítání gradientů, tedka netrénuju
        for imgs, targets in test_loader:
            imgs = imgs.to(device)
            gt_boxes = targets["box"].to(device)

            preds = model(imgs)

            batch_ious = []
            for pred_box, gt_box in zip(preds, gt_boxes):
                iou = compute_iou(
                    pred_box.cpu(),
                    gt_box.cpu()
                )
                batch_ious.append(iou)

            running_iou += sum(batch_ious) / len(batch_ious)

    avg_loss = running_loss / len(loader)
    avg_iou = running_iou / len(test_loader)

    print(
        f"Epoch {epoch+1}, "
        f"Loss: {avg_loss:.4f}, "
        f"Mean IoU: {avg_iou:.4f}"
    ) # toto taky upravený

Epoch 1, Loss: 0.0937, Mean IoU: 0.0124
Epoch 2, Loss: 0.0671, Mean IoU: 0.0432
Epoch 3, Loss: 0.0479, Mean IoU: 0.0909


In [25]:
.
# For example:
backbone_model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model = ResNetBoxes(backbone_model, freeze=False).to(device)
#   ... retrain and evaluate ...
# Also try toggling `freeze=True` vs `freeze=False` — how does freezing the backbone affect convergence and final performance?

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\marke/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:02<00:00, 21.5MB/s]
